###### <mark>**Governance – Step 1: Role-Based Access Simulation**</mark>

###### <mark>**STEP 1: Create a reusable access function**</mark>

In [1]:
from pyspark.sql.functions import col

def apply_role_based_filter(df, role):
    
    if role == "sales_team":
        return df.filter(col("country") == "in")
    
    elif role == "finance_team":
        return df
    
    elif role == "admin_team":
        return df
    
    else:
        return df.limit(0)  # no access

StatementMeta(, a00798c2-f096-4dec-85c9-3c2535af395d, 3, Finished, Available, Finished, False)

###### <mark>**STEP 2: Apply on your data**</mark>

In [2]:
df = spark.table("fact_sales").join(
    spark.table("dim_customer"),
    "customer_id"
)

StatementMeta(, a00798c2-f096-4dec-85c9-3c2535af395d, 4, Finished, Available, Finished, False)

<mark>**Test as sales user**</mark>

In [3]:
sales_df = apply_role_based_filter(df, "sales_team")
sales_df.select("country").distinct().show()

StatementMeta(, a00798c2-f096-4dec-85c9-3c2535af395d, 5, Finished, Available, Finished, False)

+-------+
|country|
+-------+
|     in|
+-------+



###### <mark>**Test as finance user**</mark>

In [4]:
finance_df = apply_role_based_filter(df, "finance_team")
finance_df.select("country").distinct().show()

StatementMeta(, a00798c2-f096-4dec-85c9-3c2535af395d, 6, Finished, Available, Finished, False)

+-------+
|country|
+-------+
|     us|
|     in|
|     au|
|     uk|
|     de|
+-------+



In [5]:
sales_df.select("country").distinct().show()
finance_df.select("country").distinct().show()

StatementMeta(, a00798c2-f096-4dec-85c9-3c2535af395d, 7, Finished, Available, Finished, False)

+-------+
|country|
+-------+
|     in|
+-------+

+-------+
|country|
+-------+
|     us|
|     in|
|     au|
|     uk|
|     de|
+-------+



##### <mark>**Next Governance Step: Column-Level Masking (PII Protection)**</mark>
**What we will protect**

**A good candidate in your project is:
email**

**Because email is personally identifiable information.**

**Use case**
  <u>Role	Email visibility</u>
- sales_team	masked email
- finance_team	full email
- admin_team	full email

<mark>**Step 1: Create masking function**</mark>

In [5]:
from pyspark.sql.functions import col, expr, concat, lit, substring, locate

def apply_column_masking(df, role):
    
    if role == "sales_team":
        return df.withColumn(
            "email",
            concat(
                substring(col("email"), 1, 1),
                lit("*****"),
                expr("substring(email, locate('@', email), length(email))")
            )
        )
    
    elif role == "finance_team":
        return df
    
    elif role == "admin_team":
        return df
    
    else:
        return df.drop("email")

StatementMeta(, 4e22a8c0-8187-4a9c-973c-d470fb42dbb7, 7, Finished, Available, Finished, False)

<mark>**Step 2: Prepare customer dataset**</mark>

In [6]:
customer_df = spark.table("dim_customer")

StatementMeta(, 4e22a8c0-8187-4a9c-973c-d470fb42dbb7, 8, Finished, Available, Finished, False)

<mark>**Step 3: Test for sales user**</mark>

In [7]:
masked_sales_customer_df = apply_column_masking(customer_df, "sales_team")
masked_sales_customer_df.select("customer_id", "customer_full_name", "email").show(10, truncate=False)

StatementMeta(, 4e22a8c0-8187-4a9c-973c-d470fb42dbb7, 9, Finished, Available, Finished, False)

+-----------+------------------+------------------+
|customer_id|customer_full_name|email             |
+-----------+------------------+------------------+
|17         |Joseph Rodriguez  |y*****@example.net|
|35         |Sue Mendez        |m*****@example.com|
|37         |Deborah Mueller   |j*****@example.com|
|41         |Michael Bass      |s*****@example.org|
|43         |Ryan Mccoy        |y*****@example.net|
|61         |Patricia Foster   |m*****@example.com|
|72         |Steven Wilson     |m*****@example.org|
|88         |Jacob Whitney     |r*****@example.org|
|107        |Sheila Evans      |l*****@example.net|
|112        |Justin Hunter     |n*****@example.net|
+-----------+------------------+------------------+
only showing top 10 rows

